# 07 — Report Generator

**Module notebook — definitions only.**

Turns the pipeline's Markdown-ish output (title, summary, action items, decisions, questions) into a clean, professional PDF — with proper right-to-left Arabic support (letter shaping + bidi reordering), not just `print()`.

Depends on: nothing upstream, but reads the same `result` dict produced by `run_pipeline()`.


In [ ]:
import os
import re
import urllib.request

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_RIGHT, TA_LEFT
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

import arabic_reshaper
from bidi.algorithm import get_display

# Professional accent palette, used for headings, rules, and the footer.
ACCENT_COLOR = HexColor("#1F4E79")   # navy — titles, section headers, divider lines
MUTED_COLOR = HexColor("#6B7280")    # gray — subtitle line, page-number footer


## 1. Arabic font

Reportlab's built-in fonts have no Arabic glyphs, so we fetch a proper Arabic typeface (Amiri, OFL-licensed) once and register it. Cached locally after the first run.

In [ ]:
FONT_DIR = "fonts"
FONT_URLS = {
    "Amiri-Regular.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Regular.ttf",
    "Amiri-Bold.ttf": "https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Bold.ttf",
}


def ensure_arabic_fonts():
    os.makedirs(FONT_DIR, exist_ok=True)
    for filename, url in FONT_URLS.items():
        path = os.path.join(FONT_DIR, filename)
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)

    if "Amiri" not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont("Amiri", os.path.join(FONT_DIR, "Amiri-Regular.ttf")))
        pdfmetrics.registerFont(TTFont("Amiri-Bold", os.path.join(FONT_DIR, "Amiri-Bold.ttf")))


## 2. Arabic shaping helper

Arabic letters change shape depending on their neighbors, and the visual reading order is right-to-left while Unicode stores it logically left-to-right. `arabic_reshaper` handles the letter-joining, `python-bidi` handles the reordering. We wrap text into lines *before* reshaping so long paragraphs still break in the right place.

In [ ]:
def shape_arabic(line: str) -> str:
    """Reshape + bidi-reorder a line of Arabic (or mixed Arabic/English) text.

    Two things this deliberately does NOT do, both fixed from an earlier version:
    - No manual fixed-width pre-wrapping. Platypus's own Paragraph layout already
      wraps text to the real column width; pre-chopping at a fixed character count
      produced ragged line breaks that didn't match the actual page width.
    - Any bullet marker must already be part of `line` *before* this function
      reorders it. Adding a bullet AFTER reordering puts it on the wrong visual
      side of the line (this was the stray "•" floating at the end of lines).
    """
    if not line.strip():
        return ""
    return get_display(arabic_reshaper.reshape(line))


## 3. Markdown → PDF flowables

Converts the LLM's lightweight Markdown (`**bold**`, `* bullets`, `1. numbered headers`) into ReportLab paragraphs.

Note on Arabic: combining inline bold *and* correct RTL shaping in the same run is unreliable with ReportLab's engine, so for Arabic we simplify — a line that's *entirely* bold becomes a section heading, and inline `**emphasis**` inside bullets is unwrapped to plain text. English keeps full inline bold.

In [ ]:
def _wrap_and_reorder(logical_text: str, font_name: str, font_size: float, max_width: float) -> list:
    """Word-wrap `logical_text` (in normal reading order) to fit `max_width`,
    measuring real glyph widths, THEN bidi-reorder each finished physical
    line for display.

    This order matters. The previous version reshaped+bidi-reordered the
    whole paragraph first and let ReportLab's Paragraph wrap the result —
    but Paragraph wraps assuming left-to-right logical text, so wrapping an
    already-reordered (visual-order) string scrambles word order on any line
    long enough to wrap. That's what produced the "غير منظم" output: short
    lines (title, labels) looked fine, long paragraphs did not.

    Wrapping first in logical order, then reordering each physical line
    independently, is the correct sequence for RTL text with a layout engine
    that isn't bidi-aware.
    """
    words = logical_text.split(" ")
    lines, current = [], ""
    for word in words:
        candidate = f"{current} {word}".strip()
        # Measure the *reshaped* candidate — reshaping can change glyph
        # widths slightly (joined letter forms), so this is more accurate
        # than measuring the raw characters.
        width = pdfmetrics.stringWidth(arabic_reshaper.reshape(candidate), font_name, font_size)
        if width <= max_width or not current:
            current = candidate
        else:
            lines.append(current)
            current = word
    if current:
        lines.append(current)

    return [get_display(arabic_reshaper.reshape(line)) for line in lines]


def markdown_to_flowables(text: str, language: str = "english", max_width: float = None) -> list:
    is_ar = language == "arabic"
    styles = getSampleStyleSheet()

    base_font = "Amiri" if is_ar else "Helvetica"
    bold_font = "Amiri-Bold" if is_ar else "Helvetica-Bold"
    align = TA_RIGHT if is_ar else TA_LEFT
    indent_kwarg = "rightIndent" if is_ar else "leftIndent"
    BULLET_INDENT = 14

    body_style = ParagraphStyle(
        "body", parent=styles["Normal"], fontName=base_font,
        alignment=align, fontSize=11, leading=18, spaceAfter=8,
    )
    bullet_style = ParagraphStyle(
        "bullet", parent=body_style, **{indent_kwarg: BULLET_INDENT},
    )
    heading_style = ParagraphStyle(
        "heading", parent=styles["Heading2"], fontName=bold_font,
        alignment=align, fontSize=13.5, leading=19,
        spaceBefore=14, spaceAfter=8, textColor=ACCENT_COLOR,
    )

    flowables = []
    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line or re.fullmatch(r"[=\-_]{3,}", line):
            continue

        is_bullet = bool(re.match(r"^[\*\-]\s+", line))
        if is_bullet:
            line = re.sub(r"^[\*\-]\s+", "", line)

        is_heading = bool(re.fullmatch(r"\*\*(.+?)\*\*:?", line)) or bool(
            re.match(r"^\d+\.\s+\*\*.+\*\*", line)
        )
        # A bold mini-heading that also had a bullet marker should render as a
        # heading only — not a heading with a stray bullet glued onto it.
        is_bullet = is_bullet and not is_heading

        style = heading_style if is_heading else (bullet_style if is_bullet else body_style)

        if is_ar:
            clean = re.sub(r"\*\*(.+?)\*\*", r"\1", line)
            # Prepend the bullet to the LOGICAL string — this is what keeps
            # the bullet on the correct (right) side after bidi reordering.
            logical = ("• " + clean) if is_bullet else clean

            if max_width is None:
                # No width to wrap against — fall back to the old
                # single-shot behavior (used only if a caller doesn't pass
                # a column width).
                flowables.append(Paragraph(shape_arabic(logical), style))
            else:
                # Headings can wrap too (long extracted action items are
                # sometimes marked up as a bold mini-heading) — never skip
                # the wrap-then-reorder pass, or a long heading hits the
                # exact same scrambling bug as long body paragraphs.
                available = max_width - (BULLET_INDENT if is_bullet else 0)
                bold_font_for_line = bold_font if is_heading else base_font
                for display_line in _wrap_and_reorder(logical, bold_font_for_line, style.fontSize, available):
                    flowables.append(Paragraph(display_line, style))
        else:
            html_line = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", line)
            content = ("• " + html_line) if is_bullet else html_line
            flowables.append(Paragraph(content, style))

    return flowables


## 4. Full report builder

Assembles title, summary, action items, key decisions and open questions into one PDF, with section labels in the right language.

In [ ]:
SECTION_LABELS = {
    "english": {
        "summary": "Summary",
        "action_items": "Action Items",
        "key_decisions": "Key Decisions",
        "open_questions": "Open Questions",
    },
    "arabic": {
        "summary": "الملخص",
        "action_items": "المهام المطلوبة",
        "key_decisions": "القرارات الرئيسية",
        "open_questions": "الأسئلة المفتوحة",
    },
    "hinglish": {
        "summary": "Summary",
        "action_items": "Action Items",
        "key_decisions": "Key Decisions",
        "open_questions": "Open Questions",
    },
}

SUBTITLE_TEXT = {
    "english": "AI-generated meeting report",
    "hinglish": "AI-generated meeting report",
    "arabic": "تقرير اجتماع تم إنشاؤه تلقائيًا",
}

TRANSCRIPT_TITLE_TEXT = {
    "english": "Full Transcript",
    "hinglish": "Full Transcript",
    "arabic": "النص الكامل",
}

TRANSCRIPT_SUBTITLE_TEXT = {
    "english": "AI-generated transcript",
    "hinglish": "AI-generated transcript",
    "arabic": "نص مُفرَّغ تم إنشاؤه تلقائيًا",
}

# PAGE_MARGIN_CM must match the margins used to build `doc` below — it's how
# markdown_to_flowables() knows the real usable column width to wrap against.
PAGE_MARGIN_CM = 2.2

# SimpleDocTemplate's default Frame adds its own left/right padding (6pt each
# side) on top of the page margins — our own pre-wrap pass needs to match
# that real usable width, or lines come out a hair too wide and ReportLab
# silently re-wraps them itself, splitting the trailing bullet/word onto its
# own line (this was the exact cause of the stray-bullet artifact).
_FRAME_PADDING = 6 + 6
# Small extra cushion for font-metric rounding differences between our
# stringWidth measurement pass and ReportLab's own layout pass.
_WRAP_SAFETY_MARGIN = 4


def _footer(canvas, doc):
    """Simple centered page-number footer, drawn on every page."""
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_COLOR)
    canvas.drawCentredString(doc.pagesize[0] / 2, 1.2 * cm, f"Page {doc.page}")
    canvas.restoreState()


def _usable_width() -> float:
    return A4[0] - 2 * (PAGE_MARGIN_CM * cm) - _FRAME_PADDING - _WRAP_SAFETY_MARGIN


def _build_header_styles(is_ar: bool):
    align = TA_RIGHT if is_ar else TA_LEFT
    title_font = "Amiri-Bold" if is_ar else "Helvetica-Bold"
    subtitle_font = "Amiri" if is_ar else "Helvetica"

    title_style = ParagraphStyle(
        "title", fontName=title_font, fontSize=20, leading=26, alignment=align,
        textColor=ACCENT_COLOR, spaceAfter=6,
    )
    subtitle_style = ParagraphStyle(
        "subtitle", fontName=subtitle_font, fontSize=9.5, leading=13, alignment=align,
        textColor=MUTED_COLOR, spaceAfter=14,
    )
    return title_style, subtitle_style


def generate_pdf_report(result: dict, output_path: str = "meeting_report.pdf", language: str = "english") -> str:
    is_ar = language == "arabic"
    if is_ar:
        ensure_arabic_fonts()

    labels = SECTION_LABELS.get(language, SECTION_LABELS["english"])
    align = TA_RIGHT if is_ar else TA_LEFT
    rule_align = "RIGHT" if is_ar else "LEFT"
    section_font = "Amiri-Bold" if is_ar else "Helvetica-Bold"

    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        topMargin=PAGE_MARGIN_CM * cm, bottomMargin=2 * cm,
        leftMargin=PAGE_MARGIN_CM * cm, rightMargin=PAGE_MARGIN_CM * cm,
    )
    max_width = _usable_width()

    title_style, subtitle_style = _build_header_styles(is_ar)
    section_style = ParagraphStyle(
        "section", fontName=section_font, fontSize=14, leading=18, alignment=align,
        textColor=ACCENT_COLOR, spaceBefore=16, spaceAfter=4,
    )

    story = []

    # --- Header: title, subtitle, full-width accent rule ---
    title_text = shape_arabic(result.get("title", "")) if is_ar else (result.get("title") or "Untitled Recording")
    story.append(Paragraph(title_text or "Untitled Recording", title_style))

    subtitle_raw = SUBTITLE_TEXT.get(language, SUBTITLE_TEXT["english"])
    subtitle_text = shape_arabic(subtitle_raw) if is_ar else subtitle_raw
    story.append(Paragraph(subtitle_text, subtitle_style))
    story.append(HRFlowable(width="100%", thickness=1.2, color=ACCENT_COLOR, spaceAfter=12))

    # --- Sections, each with a short accent rule under its heading ---
    sections = [
        ("summary", result.get("summary", "")),
        ("action_items", result.get("action_items", "")),
        ("key_decisions", result.get("key_decisions", "")),
        ("open_questions", result.get("open_questions", "")),
    ]

    for key, content in sections:
        if not content or not str(content).strip():
            continue
        label = shape_arabic(labels[key]) if is_ar else labels[key]
        story.append(Paragraph(label, section_style))
        story.append(HRFlowable(width="25%", thickness=0.8, color=ACCENT_COLOR, hAlign=rule_align, spaceAfter=8))
        story += markdown_to_flowables(str(content), language, max_width=max_width)

    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)
    return output_path


def generate_transcript_pdf(transcript: str, title: str = "", output_path: str = "transcript.pdf", language: str = "english") -> str:
    """Same styling/RTL handling as generate_pdf_report, but for the raw
    transcript alone — used by the Streamlit transcript tab's download button."""
    is_ar = language == "arabic"
    if is_ar:
        ensure_arabic_fonts()

    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        topMargin=PAGE_MARGIN_CM * cm, bottomMargin=2 * cm,
        leftMargin=PAGE_MARGIN_CM * cm, rightMargin=PAGE_MARGIN_CM * cm,
    )
    max_width = _usable_width()
    title_style, subtitle_style = _build_header_styles(is_ar)

    story = []

    header_text = title.strip() if title and title.strip() else TRANSCRIPT_TITLE_TEXT.get(language, TRANSCRIPT_TITLE_TEXT["english"])
    header_text = shape_arabic(header_text) if is_ar else header_text
    story.append(Paragraph(header_text, title_style))

    subtitle_raw = TRANSCRIPT_SUBTITLE_TEXT.get(language, TRANSCRIPT_SUBTITLE_TEXT["english"])
    subtitle_text = shape_arabic(subtitle_raw) if is_ar else subtitle_raw
    story.append(Paragraph(subtitle_text, subtitle_style))
    story.append(HRFlowable(width="100%", thickness=1.2, color=ACCENT_COLOR, spaceAfter=12))

    # The transcript is one long block of prose with no markdown structure,
    # but it can still contain paragraph breaks — treat each non-empty line
    # as its own paragraph and reuse the same wrap-then-reorder path.
    story += markdown_to_flowables(str(transcript or ""), language, max_width=max_width)

    doc.build(story, onFirstPage=_footer, onLaterPages=_footer)
    return output_path
